In [ ]:
import pandas as pd

In [ ]:
games = pd.read_csv("nba_games_2019to2025.csv",index_col=0)
games = games.drop(columns=['mp.1','mp_opp.1'])


In [ ]:
object_cols = games.dtypes[games.dtypes == 'object']
print(object_cols)

In [ ]:
games["date"] = pd.to_datetime(games["date"])

In [ ]:
games["opp_code"] = games["team_opp"].astype("category").cat.codes

In [ ]:
games["target"] = (games["won"] == True).astype("int")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(n_estimators=5000,min_samples_split=100,random_state=1)

In [ ]:
games['month'] = games['date'].dt.month

In [ ]:
train = games[games["date"] < "2024-09-01"]
test = games[games["date"] >"2024-09-01"]

In [ ]:
predictors = ["opp_code","home","month"]

In [ ]:
rf.fit(train[predictors],train["target"])

In [ ]:
pred = rf.predict(test[predictors])

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
acc = accuracy_score(test["target"],pred)

In [ ]:
combined = pd.DataFrame(dict(actual = test["target"],prediction = pred))
pd.crosstab(index=combined["actual"], columns = combined["prediction"])

In [ ]:
from sklearn.metrics import precision_score
precision_score(test["target"],pred)

In [ ]:
grouped_games = games.groupby("team")

In [ ]:
def rolling_avg(group, cols, new_cols):
    group = group.sort_values("date")
    rolling_stats = group[cols].rolling(15,closed='left').mean()
    group[new_cols] = rolling_stats
    group = group.dropna(subset = new_cols)
    return group

In [ ]:
cols = ['fg','fga','fg%','3pa','3p','3p%','ft','fta','ft%','orb','drb','ast','tov','ts%',
        'fg_opp','fga_opp','fg%_opp','3pa_opp','3p_opp','3p%_opp','ft_opp','fta_opp','ft%_opp','orb_opp','drb_opp','ast_opp','tov_opp','ts%_opp']

new_cols = [f"{c}_rolling" for c in cols]

In [ ]:
games_rolling = games.groupby("team").apply(lambda x: rolling_avg(x,cols,new_cols))

In [ ]:
games_rolling = games_rolling.droplevel("team")
games_rolling

In [ ]:
games_rolling.index = range(games_rolling.shape[0]) #unique val for each indx

In [ ]:
def make_preds(data,predictors):
    train = data[data["date"] < "2024-09-01"]
    test = data[data["date"].between("2024-09-01", "2025-04-19", inclusive='neither')]
    rf.fit(train[predictors],train["target"])
    pred = rf.predict(test[predictors])
    combined = pd.DataFrame(dict(actual = test["target"],prediction = pred),index=test.index)
    precision = precision_score(test["target"],pred)
    acc = accuracy_score(test["target"],pred)
    return combined,precision,acc


     
    

In [ ]:
combined, precision, acc = make_preds(games_rolling, predictors + new_cols)


In [417]:
precision

0.624262847514743

In [418]:
acc

0.6196895424836601

In [ ]:
combined = combined.merge(games_rolling[["date","team","team_opp","won"]],left_index=True,right_index=True)

In [ ]:
predicted_wins = (
    combined.groupby(["conference","team"], as_index=False)["prediction"]
            .sum()
            .rename(columns={"prediction":"predicted_wins"})
            .sort_values(["conference","predicted_wins"], ascending=[True, False])
)


In [416]:
# Count wins and sort
# Define conferences
eastern_conf = ['ATL', 'BOS', 'BRK', 'CHO', 'CHI', 'CLE', 'DET', 'IND', 
                'MIA', 'MIL', 'NYK', 'ORL', 'PHI', 'TOR', 'WAS']

western_conf = ['DAL', 'DEN', 'GSW', 'HOU', 'LAC', 'LAL', 'MEM', 'MIN', 
                'NOP', 'OKC', 'PHO', 'POR', 'SAC', 'SAS', 'UTA']

# Add conference column 
def assign_conference(team):
    if team in eastern_conf:
        return 'East'
    else:
        return 'West'
   

combined['conference'] = combined['team'].apply(assign_conference)

# East
east = predicted_wins[predicted_wins['conference'] == 'East'].reset_index(drop=True)
east.index = east.index + 1
east.index.name = 'Rank'

# West
west = predicted_wins[predicted_wins['conference'] == 'West'].reset_index(drop=True)
west.index = west.index + 1
west.index.name = 'Rank'

print("EASTERN CONFERENCE 2024-2025:")
print(east)

print("\nWESTERN CONFERENCE 2024-2025:")
print(west)

EASTERN CONFERENCE 2024-2025:
     conference team  predicted_wins
Rank                                
1          East  BOS              82
2          East  CLE              75
3          East  MIL              61
4          East  IND              58
5          East  NYK              55
6          East  MIA              48
7          East  DET              42
8          East  CHI              41
9          East  ORL              40
10         East  TOR              18
11         East  PHI              15
12         East  BRK              12
13         East  ATL               8
14         East  CHO               1
15         East  WAS               1

WESTERN CONFERENCE 2024-2025:
     conference team  predicted_wins
Rank                                
1          West  OKC              79
2          West  MIN              59
3          West  LAC              58
4          West  HOU              57
5          West  PHO              56
6          West  DEN              55
7          Wes